<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula13a_titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!unzip titanic.zip

Archive:  titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [2]:
import pandas as pd

df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [4]:
y = df['Survived']
X = df.drop('Survived', axis=1)

In [5]:
len(y)

891

In [6]:
for col in X.columns:
  print(f"{col:>12} {len(set(X[col])):>4} {X[col].dtype}")

 PassengerId  891 int64
      Pclass    3 int64
        Name  891 object
         Sex    2 object
         Age  265 float64
       SibSp    7 int64
       Parch    7 int64
      Ticket  681 object
        Fare  248 float64
       Cabin  148 object
    Embarked    4 object


In [7]:
caracteristicas_indesejadas = ['PassengerId', 'Name', 'Ticket', 'Cabin']
Xdrop = X.drop(caracteristicas_indesejadas, axis=1)
Xdrop.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [8]:
Xnum = Xdrop.select_dtypes('number')
Xnum.columns

Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')

In [9]:
for col in Xnum.columns:
  print(f"{col:>12} {Xnum[col].isnull().sum():>4}")

      Pclass    0
         Age  177
       SibSp    0
       Parch    0
        Fare    0


In [10]:
Xnum[-5:]

,Pclass,Age,SibSp,Parch,Fare
886,2,27.0,0,0,13.00
887,1,19.0,0,0,30.00
888,3,NaN,1,2,23.45
889,1,26.0,0,0,30.00
890,3,32.0,0,0,7.75


In [13]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
XnumTratado = imputer.fit_transform(Xnum)
XnumTratado[-5:]

array([[ 2.  , 27.  ,  0.  ,  0.  , 13.  ],
       [ 1.  , 19.  ,  0.  ,  0.  , 30.  ],
       [ 3.  , 28.  ,  1.  ,  2.  , 23.45],
       [ 1.  , 26.  ,  0.  ,  0.  , 30.  ],
       [ 3.  , 32.  ,  0.  ,  0.  ,  7.75]])

In [14]:
Xcat = Xdrop.select_dtypes(object)
Xcat.columns

Index(['Sex', 'Embarked'], dtype='object')

In [15]:
for col in Xcat.columns:
  print(f"{col:>12} {Xcat[col].isnull().sum():>4}")

         Sex    0
    Embarked    2


In [17]:
Xcat[-5:]

,Sex,Embarked
886,male,S
887,female,S
888,female,S
889,male,C
890,male,Q


In [16]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')
XcatTratado = imputer.fit_transform(Xcat)
XcatTratado[-5:]

array([['male', 'S'],
       ['female', 'S'],
       ['female', 'S'],
       ['male', 'C'],
       ['male', 'Q']], dtype=object)

In [18]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
XcatTratadoHot = encoder.fit_transform(XcatTratado)
XcatTratadoHot

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1782 stored elements and shape (891, 5)>

In [19]:
XcatTratadoHot.toarray()[-5:]

array([[0., 1., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.]])

In [22]:
import numpy as np

XTratado = np.c_[XnumTratado, XcatTratadoHot.toarray()]
print(XTratado.shape)
XTratado[-5:]

(891, 10)


array([[ 2.  , 27.  ,  0.  ,  0.  , 13.  ,  0.  ,  1.  ,  0.  ,  0.  ,
         1.  ],
       [ 1.  , 19.  ,  0.  ,  0.  , 30.  ,  1.  ,  0.  ,  0.  ,  0.  ,
         1.  ],
       [ 3.  , 28.  ,  1.  ,  2.  , 23.45,  1.  ,  0.  ,  0.  ,  0.  ,
         1.  ],
       [ 1.  , 26.  ,  0.  ,  0.  , 30.  ,  0.  ,  1.  ,  1.  ,  0.  ,
         0.  ],
       [ 3.  , 32.  ,  0.  ,  0.  ,  7.75,  0.  ,  1.  ,  0.  ,  1.  ,
         0.  ]])

In [25]:
from sklearn.base import BaseEstimator, TransformerMixin

class AtributosDesejados(BaseEstimator, TransformerMixin):
  def __init__(self):
    self.colunas_indesejadas = ['PassengerId', 'Name', 'Ticket', 'Cabin']
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.drop(self.colunas_indesejadas, axis=1)

atributos_desejados = AtributosDesejados()
Xdrop = atributos_desejados.fit_transform(X)
Xdrop.tail()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
886,2,male,27.0,0,0,13.00,S
887,1,female,19.0,0,0,30.00,S
888,3,female,NaN,1,2,23.45,S
889,1,male,26.0,0,0,30.00,C
890,3,male,32.0,0,0,7.75,Q


In [26]:
class AtributosNumericos(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.select_dtypes(include='number')

atributos_numericos = AtributosNumericos()
Xnum = atributos_numericos.fit_transform(Xdrop)
Xnum.tail()

,Pclass,Age,SibSp,Parch,Fare
886,2,27.0,0,0,13.00
887,1,19.0,0,0,30.00
888,3,NaN,1,2,23.45
889,1,26.0,0,0,30.00
890,3,32.0,0,0,7.75


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipenum = Pipeline([
    ('atributos_numericos', AtributosNumericos()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
Xnum = pipenum.fit_transform(Xdrop)
Xnum[-5:]

array([[-0.36936484, -0.18148724, -0.4745452 , -0.47367361, -0.38667072],
       [-1.56610693, -0.79628599, -0.4745452 , -0.47367361, -0.04438104],
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783]])

In [29]:
class AtributosCategoricos(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.select_dtypes(include='object')

atributos_categoricos = AtributosCategoricos()
Xcat = atributos_categoricos.fit_transform(Xdrop)
Xcat.tail()

,Sex,Embarked
886,male,S
887,female,S
888,female,S
889,male,C
890,male,Q


In [31]:
pipecat = Pipeline([
    ('atributos_categoricos', AtributosCategoricos()),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())
])
XcatTratado = pipecat.fit_transform(Xdrop)
XcatTratado.toarray()[-5:]

array([[0., 1., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.]])

In [33]:
from sklearn.pipeline import FeatureUnion

unecaracteristicas = FeatureUnion([
    ('pipenum', pipenum),
    ('pipecat', pipecat)
])
Xtratado = unecaracteristicas.fit_transform(Xdrop)
Xtratado[-5:]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 35 stored elements and shape (5, 10)>

In [35]:
class DenseTransformer(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.toarray()

preproc = Pipeline([
    ('atributos_desejados', AtributosDesejados()),
    ('unecaracteristicas', unecaracteristicas),
    ('to_dense', DenseTransformer())
])
Xtratado = preproc.fit_transform(X)
Xtratado[-5:]

array([[-0.36936484, -0.18148724, -0.4745452 , -0.47367361, -0.38667072,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.79628599, -0.4745452 , -0.47367361, -0.04438104,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104,
         0.        ,  1.        ,  1.        ,  0.        ,  0.        ],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ]])

In [36]:
from sklearn.ensemble import RandomForestClassifier

model = Pipeline([
    ('preproc', preproc),
    ('classificador', RandomForestClassifier())
])
model.fit(X, y)

Pipeline(steps=[('preproc',
                 Pipeline(steps=[('atributos_desejados', AtributosDesejados()),
                                 ('unecaracteristicas',
                                  FeatureUnion(transformer_list=[('pipenum',
                                                                  Pipeline(steps=[('atributos_numericos',
                                                                                   AtributosNumericos()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='median')),
                                                                                  ('scaler',
                                                                                   StandardScaler())])),
                                                                 ('pipecat',
                                                                  Pipeline(steps=[('atributos_categoricos',
                                                                                   AtributosCategoricos()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='most_frequent')),
                                                                                  ('encoder',
                                                                                   OneHotEncoder())]))])),
                                 ('to_dense', DenseTransformer())])),
                ('classificador', RandomForestClassifier())])

In [38]:
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

y_pred = model.predict(X)
accuracy_score(y, y_pred)

0.9797979797979798

In [39]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
print(scores.mean())
print(scores)

0.8125918021467579
[0.79329609 0.82022472 0.85955056 0.7752809  0.81460674]


In [40]:
Xtest = pd.read_csv('test.csv')
y_pred = model.predict(Xtest)
y_pred

array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [41]:
!head gender_submission.csv

PassengerId,Survived
892,0
893,1
894,0
895,0
896,1
897,0
898,1
899,0
900,1


In [42]:
submission = pd.read_csv('gender_submission.csv')
submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [43]:
submission['Survived'] = y_pred
submission.to_csv('rf_submission.csv', index=False)

In [44]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

estimators = {
    'LogisticRegression': LogisticRegression(),
    'SGDClassifier': SGDClassifier(),
    'KNeighborsClassifier': KNeighborsClassifier(),
    'SVC': SVC(),
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'RandomForestClassifier': RandomForestClassifier(),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'AdaBoostClassifier': AdaBoostClassifier()
}

results = {
    'estimator': [],
    'accuracy_mean': [],
    'accuracy_std': []
}

for estimator in estimators:
  model = Pipeline([
      ('preproc', preproc),
      ('classificador', estimators[estimator])
  ])
  scores = cross_val_score(model, X, y, cv=5)
  results['estimator'].append(estimator)
  results['accuracy_mean'].append(scores.mean())
  results['accuracy_std'].append(scores.std())
  print(f"{estimator:>30}: {scores.mean():.2f} +/- {scores.std():.2f}")


            LogisticRegression: 0.79 +/- 0.02
                 SGDClassifier: 0.77 +/- 0.04
          KNeighborsClassifier: 0.81 +/- 0.02
                           SVC: 0.83 +/- 0.02
        DecisionTreeClassifier: 0.78 +/- 0.03
        RandomForestClassifier: 0.80 +/- 0.03
    GradientBoostingClassifier: 0.82 +/- 0.02
            AdaBoostClassifier: 0.80 +/- 0.02


In [46]:
!pip install optuna -qq

In [ ]:
from multiprocessing import Pipe
from sklearn.base import BaseEstimator, ClassifierMixin
import optuna

class OptunaSVC(BaseEstimator, TransformerMixin):
  def __init__(self, trial=5):
    self.trial = trial
  def fit(self, X, y=None):
    def objective(trial):
      c = trial.suggest_float('C', 1e-10, 1e10, log=True)
      gamma = trial.suggest_float('gamma', 1e-10, 1e10, log=True)
      model = Pipeline([
          ('preproc', preproc),
          ('classificador', SVC(C=c, gamma=gamma))
      ])
      scores = cross_val_score(model, X, y, cv=5)
      return scores.mean()
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=self.trial)
    self.model = Pipeline([
        ('preproc', preproc),
        ('classificador', SVC(**study.best_params))
    ])
    self.model.fit(X, y)
    return self
  def predict(self, X, y=None):
    return self.model.predict(X)
  def score(self, X, y):
    return self.model.score(X, y)

model = OptunaSVC(trial=20)
scores = cross_val_score(model, X, y, cv=5)
print(scores.mean())
print(scores)

[I 2026-09-29 15:52:18,573] A new study created in memory with name: no-name-c7ce353d-a89f-4671-aacf-f9c30fa2b118
